In [ ]:
from llama_cpp import Llama
import pandas as pd
import json
import re
from tqdm import tqdm

# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"


llm = Llama.from_pretrained(
    repo_id="bartowski/JSL-MedLlama-3-8B-v2.0-GGUF",
    filename="JSL-MedLlama-3-8B-v2.0-Q4_K_M.gguf",  # you can pick other quants too

    n_gpu_layers=-1,
    n_threads=4,    
    n_batch=256,

    use_mmap=False,
    use_mlock=True
)

PROMPT = """
You are a clinical information extraction system.

Extract the drug names and adverse drug events (ADEs) from the text below.

Return ONLY valid JSON in this exact format:

{{
  "drug_names": ["drug1", "drug2"],
  "adverse_effects": ["effect1", "effect2"]
}}

Text:
{text}

ONLY RETURN THE JSON. NO EXTRA TEXT.
"""

def trim_text_for_context(text, max_chars=1200):
    if len(text) <= max_chars:
        return text
    return text[:max_chars]

def extract_drug_adr(text):
    trimmed = trim_text_for_context(text)
    prompt = PROMPT.format(text=trimmed)

    result = llm(
        prompt=prompt,
        max_tokens=256,
        temperature=0.0
    )

    raw_output = result["choices"][0]["text"].strip()
    cleaned = raw_output.replace("```json", "").replace("```", "").strip()
    matches = re.findall(r"\{[\s\S]*?\}", cleaned)
    if matches:
        cleaned = matches[-1]

    try:
        print(cleaned)
        return json.loads(cleaned)
    except Exception:
        print("\n⚠️ JSON parse failed for text:", trimmed)
        print("Model output:", raw_output)
        print("Cleaned JSON:", cleaned)
        return {"drug_names": [], "adverse_effects": []}

INPUT_CSV = "./data/LLaVA-Med/subset_of_all_ADR.csv"
OUTPUT_CSV = "./data/LLaVA-Med/output_medllama_adr.csv"

df = pd.read_csv(INPUT_CSV)

drug_col = []
adr_col = []

print("\n✅ Starting extraction over CSV...\n")

for text in tqdm(df["Preprocessed Posts"], desc="Extracting ADEs"):
    res = extract_drug_adr(str(text))
    drug_col.append(", ".join(res.get("drug_names", [])))
    adr_col.append(", ".join(res.get("adverse_effects", [])))

df["drug_names"] = drug_col
df["adverse_effects"] = adr_col

df.to_csv(OUTPUT_CSV, index=False)

print("\n✅ DONE! Output saved to:", OUTPUT_CSV)
